In [1]:
import cv2
import numpy as np
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh.FaceMesh(
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

RIGHT = [33, 160, 159, 158, 133, 153, 145, 144]
LEFT  = [362, 385, 386, 387, 263, 373, 374, 380]

def ear_3d(pts):
    P0,P3,P4,P5,P8,P11,P12,P13 = pts
    n = np.linalg.norm(P3-P13)**3 + np.linalg.norm(P4-P12)**3 + np.linalg.norm(P5-P11)**3
    d = 3 * np.linalg.norm(P0-P8)**3
    return n / (d + 1e-6)

cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret: break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = mp_face_mesh.process(rgb)
    if res.multi_face_landmarks:
        lm = np.array([[p.x, p.y, p.z] for p in res.multi_face_landmarks[0].landmark])
        r = ear_3d(lm[RIGHT])
        l = ear_3d(lm[LEFT])
        ear = (r + l + 1) / 2
        color = (0, 255, 0) if ear > 0.45 else (0, 0, 255)
        cv2.putText(frame, f"EAR: {ear:.4f}", (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 2)
        cv2.putText(frame, f"R:{r:.3f} L:{l:.3f}", (30, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200,200,200), 1)
    cv2.imshow("EAR Debug", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

AttributeError: module 'mediapipe' has no attribute 'solutions'